# 12. ORDER BY

*Module: 4 — Advanced querying · Companion summary: `lectures/12-order-by.md` · SQL engine: SQLite (in-memory)*

## What you will study

- sort result rows ascending and descending
- sort by column position and by multiple columns
- combine ORDER BY with LIMIT for a Top-N query

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — BOOK

In [ ]:
q("CREATE TABLE book (book_id TEXT PRIMARY KEY, title TEXT, pages INTEGER, year INTEGER)")
q("""
INSERT INTO book VALUES
    ('B1', 'The Little Prince',   292, 1943),
    ('B2', 'Journey to the West', 298, 1592),
    ('B3', 'SQL Basics',          150, 2018),
    ('B4', 'Data Science Handbook', 400, 2020),
    ('B5', 'A Short Story',        45, 2022)
""")
q("SELECT * FROM book")

## 🧪 ORDER BY — ascending is the default

In [ ]:
q("SELECT title, pages FROM book ORDER BY pages")

## 🧪 DESC — descending

In [ ]:
q("SELECT title, pages FROM book ORDER BY pages DESC")

## 🧪 Positional ORDER BY — '2' means the 2nd column of the SELECT list

In [ ]:
q("SELECT title, pages FROM book ORDER BY 2 DESC")

## 🧪 Multiple columns — secondary sort key

In [ ]:
q("SELECT title, year, pages FROM book ORDER BY year DESC, pages ASC")
q("SELECT title, year, pages FROM book ORDER BY 2, 3")

## 🧪 Top-N — the classic ORDER BY + LIMIT combo

In [ ]:
q("SELECT title, pages FROM book ORDER BY pages DESC LIMIT 3")

## 💡 Tips & tricks

- ORDER BY runs *after* the rows are selected — it's presentation, not filtering.
- Prefer naming columns over positional ORDER BY (readability), but know the position
  form for reading other people's code.
- 📕 Dialect note: identical on all engines; SQL Server uses `TOP`/`OFFSET…FETCH`
  syntaxes but ORDER BY itself is universal.

## 🎯 Your turn

1. The 2 oldest books (smallest years), newest-first sorted... careful: sort descending
   by year and take the 2 **newest**; then sort ascending and take 2 **oldest**.
2. Alphabetical list of titles, Z→A.

In [ ]:
# 💡 Solution
q("SELECT title, year FROM book ORDER BY year DESC LIMIT 2")   # 2 newest
q("SELECT title, year FROM book ORDER BY year ASC LIMIT 2")   # 2 oldest
q("SELECT title FROM book ORDER BY title DESC")